# SDA_INDIA_0.pdf Table Extraction Pipeline

`SDA_INDIA_0.pdf` is 356 pages, ~152MB, and mixes real text/table pages with scanned image pages. Pipeline:

1. **Classify pages** — walk every page with `pdfplumber` and check whether it has real extractable text. Pages that are just images (scanned, no text layer) are skipped entirely; the two libraries below can't do anything useful with them anyway.
2. **Extract tables** — only on the text-bearing pages, run both `camelot` (lattice + stream) and `pdfplumber`, independently.
3. **Validate & restructure with OpenAI** — for each page, hand both raw extractions + the page text to the LLM and ask it to reconcile them into one clean table (columns, rows, notes on what it changed/discarded). This is a *validation layer*, not a from-scratch extractor: it never invents numbers, it only arbitrates between what camelot and pdfplumber already found.

Run step 1 (classification) over the full document first — it's cheap. Then run steps 2-3 on a small slice of the resulting text-page list before scaling up; camelot + an LLM call per page over the whole doc is slow and not free.

## 0. Setup

`camelot` needs Ghostscript on the system for lattice mode:
```bash
brew install ghostscript   # macOS
```
Then the Python deps:

In [3]:
%pip install -q "camelot-py[cv]" pdfplumber pymupdf openai python-dotenv pandas


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import io
import json
import os
import re
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
from typing import Any, Dict, List, Optional

import camelot
import pdfplumber
import pandas as pd
from dotenv import load_dotenv
import openai

import pymupdf

from pdf_extraction_workers import extract_camelot_chunk, extract_pdfplumber_chunk

load_dotenv(Path(".env"))  # backend/.env -- notebook runs with cwd=backend/

PDF_PATH = Path("../SDA_INDIA_0.pdf")
assert PDF_PATH.exists(), f"PDF not found at {PDF_PATH.resolve()}"

# Minimum non-whitespace characters of extracted text for a page to count as
# "has real text" rather than "just an image" (scan noise / a stray page
# number can still yield a handful of characters).
TEXT_MIN_CHARS = 20

# Extraction workers -- one process per page-chunk for camelot/pdfplumber.
CPU_WORKERS = min(os.cpu_count() or 4, 8)

OUTPUT_DIR = Path("data/sda_india_extraction")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OPENAI_MODEL = "gpt-4o-mini"
client = openai.OpenAI()  # reads OPENAI_API_KEY from the environment

def chunk_list(lst: List[int], n_chunks: int) -> List[List[int]]:
    """Split into up to n_chunks pieces, round-robin (lst[0::n], lst[1::n], ...)
    rather than contiguous slices. A handful of pages in this PDF (heavy vector
    graphics / large embedded images) take far longer to parse than the rest --
    with contiguous slicing those pathological pages can all land in the same
    chunk, so that one worker becomes the bottleneck and parallelism buys
    nothing. Round-robin spreads them evenly across every worker instead."""
    n_chunks = max(1, min(n_chunks, len(lst)))
    if not lst:
        return []
    return [c for i in range(n_chunks) if (c := lst[i::n_chunks])]


## 1. Classify pages: text vs image-only

Scans every page once with `pymupdf`, checking for real extractable text. Fast enough (~10s for the full 356-page document) to run over the whole document single-threaded -- see the note in `classify_pages` for why this isn't pdfplumber-based or parallelized. Pages with no meaningful text (pure scanned images, blank separators, cover pages) are dropped from every later step.

In [5]:
def classify_pages(pdf_path: Path, min_chars: int = TEXT_MIN_CHARS) -> tuple[List[int], Dict[int, str]]:
    """Returns (list of 1-based page numbers with real text, {page_num: text} for those pages).

    Uses pymupdf instead of pdfplumber here: pdfplumber/pdfminer's layout
    analysis is pathologically slow on a handful of this PDF's pages (heavy
    vector graphics), taking 5-6 minutes for the full document -- and that
    doesn't improve by parallelizing, since whichever worker gets one of
    those pages just stalls on it instead. pymupdf's C-based text extraction
    does the same full-document scan in under 10 seconds single-threaded, so
    there's no need to parallelize this step at all."""
    doc = pymupdf.open(pdf_path)
    text_pages: List[int] = []
    page_text: Dict[int, str] = {}
    for i, page in enumerate(doc, start=1):
        text = page.get_text("text")
        if len(text.strip()) >= min_chars:
            text_pages.append(i)
            page_text[i] = text
    doc.close()
    return text_pages, page_text


text_pages, page_text = classify_pages(PDF_PATH)
print(f"{len(text_pages)} page(s) have extractable text (image-only pages skipped)")


346 page(s) have extractable text (image-only pages skipped)


In [6]:
# Work on a slice of the text-bearing pages first -- widen once the pipeline
# looks right. Pass a plain list of page numbers, e.g. text_pages[:10].
TARGET_PAGES: List[int] = text_pages[:10]

def pages_to_camelot_range(pages: List[int]) -> str:
    """camelot wants a comma-separated string of individual page numbers/ranges."""
    return ",".join(str(p) for p in pages)


print(f"Running extraction on {len(TARGET_PAGES)} page(s): {TARGET_PAGES}")

Running extraction on 10 page(s): [1, 2, 3, 5, 7, 8, 9, 10, 11, 12]


### 2a. camelot
Tries `lattice` (ruled tables, needs visible grid lines) first, falls back to `stream` (whitespace-based). Runs in parallel across processes, chunking `TARGET_PAGES` across up to `CPU_WORKERS` cores.

In [7]:
def extract_with_camelot(pdf_path: Path, pages: List[int], n_workers: int = CPU_WORKERS) -> List[Dict[str, Any]]:
    """Parallel camelot extraction: pages are split into chunks, one process per
    chunk (each still runs both lattice and stream over its own pages).
    Splitting into processes lets independent pages parse on separate cores
    instead of one long sequential camelot.read_pdf call over all pages."""
    chunks = chunk_list(pages, n_workers)
    results: List[Dict[str, Any]] = []
    if not chunks:
        return results

    with ProcessPoolExecutor(max_workers=len(chunks)) as pool:
        futures = {pool.submit(extract_camelot_chunk, str(pdf_path), chunk): chunk for chunk in chunks}
        for future in as_completed(futures):
            chunk = futures[future]
            try:
                results.extend(future.result())
            except Exception as e:
                print(f"[camelot] chunk {chunk[0]}-{chunk[-1]} failed: {e}")
    return results


camelot_tables = extract_with_camelot(PDF_PATH, TARGET_PAGES)
print(f"camelot found {len(camelot_tables)} table(s)")
for t in sorted(camelot_tables, key=lambda t: t["page"]):
    print(f"  page {t['page']:>4}  {t['method']:<16}  accuracy={t['accuracy']:>5}  shape={t['df'].shape}")


camelot found 11 table(s)
  page    1  camelot_stream    accuracy=100.0  shape=(6, 2)
  page    2  camelot_stream    accuracy=100.0  shape=(10, 1)
  page    3  camelot_stream    accuracy=100.0  shape=(5, 1)
  page    5  camelot_stream    accuracy=100.0  shape=(33, 1)
  page    7  camelot_stream    accuracy=100.0  shape=(35, 1)
  page    8  camelot_stream    accuracy=100.0  shape=(4, 2)
  page    8  camelot_stream    accuracy=55.79  shape=(34, 2)
  page    9  camelot_stream    accuracy=100.0  shape=(11, 1)
  page   10  camelot_stream    accuracy=100.0  shape=(36, 1)
  page   11  camelot_stream    accuracy=100.0  shape=(31, 1)
  page   12  camelot_stream    accuracy=99.25  shape=(34, 2)


### 2b. pdfplumber
Independent extraction using pdfplumber's own table-detection heuristics (page text was already captured during classification, so it isn't re-read here). Also parallelized across processes.

In [8]:
def extract_with_pdfplumber(pdf_path: Path, pages: List[int], n_workers: int = CPU_WORKERS) -> List[Dict[str, Any]]:
    """Parallel pdfplumber table extraction, chunked the same way as camelot."""
    chunks = chunk_list(pages, n_workers)
    results: List[Dict[str, Any]] = []
    if not chunks:
        return results

    with ProcessPoolExecutor(max_workers=len(chunks)) as pool:
        futures = {pool.submit(extract_pdfplumber_chunk, str(pdf_path), chunk): chunk for chunk in chunks}
        for future in as_completed(futures):
            chunk = futures[future]
            try:
                results.extend(future.result())
            except Exception as e:
                print(f"[pdfplumber] chunk {chunk[0]}-{chunk[-1]} failed: {e}")
    return results


plumber_tables = extract_with_pdfplumber(PDF_PATH, TARGET_PAGES)
print(f"pdfplumber found {len(plumber_tables)} table(s)")
for t in sorted(plumber_tables, key=lambda t: t["page"]):
    print(f"  page {t['page']:>4}  {t['method']:<12}  shape={t['df'].shape}")


pdfplumber found 3 table(s)
  page    1  pdfplumber    shape=(2, 6)
  page    1  pdfplumber    shape=(2, 4)
  page    8  pdfplumber    shape=(2, 2)


### 2c. Group both extractions per page

In [9]:
def group_by_page(camelot_tables: List[Dict], plumber_tables: List[Dict]) -> Dict[int, Dict[str, List[Dict]]]:
    pages: Dict[int, Dict[str, List[Dict]]] = {}
    for t in camelot_tables:
        pages.setdefault(t["page"], {"camelot": [], "pdfplumber": []})["camelot"].append(t)
    for t in plumber_tables:
        pages.setdefault(t["page"], {"camelot": [], "pdfplumber": []})["pdfplumber"].append(t)
    return dict(sorted(pages.items()))


pages_grouped = group_by_page(camelot_tables, plumber_tables)
print(f"{len(pages_grouped)} page(s) have at least one table candidate: {list(pages_grouped.keys())}")

10 page(s) have at least one table candidate: [1, 2, 3, 5, 7, 8, 9, 10, 11, 12]


## 3. OpenAI validation / restructuring layer

For each page with a table candidate, both raw extractions (and the page text from step 1) are handed to the model. It is instructed to **reconcile, not invent**: pick the extraction that got a given row/column right, fix obvious splits/merges, and flag anything it's unsure about rather than guessing a number. Output is constrained to JSON (`response_format={"type": "json_object"}`).

In [10]:
def df_to_text(df: pd.DataFrame, max_rows: int = 40) -> str:
    trimmed = df.head(max_rows)
    return trimmed.to_csv(index=False, header=False)


def build_validation_prompt(page_num: int, candidates: Dict[str, List[Dict]], text: str) -> str:
    sections = []
    for t in candidates.get("camelot", []):
        sections.append(
            f"--- camelot ({t['method']}, accuracy={t.get('accuracy')}) ---\n{df_to_text(t['df'])}"
        )
    for i, t in enumerate(candidates.get("pdfplumber", [])):
        sections.append(f"--- pdfplumber table {i + 1} ---\n{df_to_text(t['df'])}")

    return f"""You are validating table extractions from page {page_num} of an Indian government survey PDF (SDA_INDIA).

You are given one or more independent extractions of the same page's table(s) by two different tools
(camelot and pdfplumber), plus the raw page text for grounding. The tools frequently disagree: one may
merge two columns, split a multi-line header across rows, drop a footnote, or misread a numeric column.

Raw page text:
{text[:3000]}

Extracted candidates:
{chr(10).join(sections) if sections else '(no table candidates extracted on this page)'}

Your task: reconcile these into ONE clean, correct table per distinct table on the page.
Rules:
- NEVER invent a value that isn't present in at least one candidate or the raw text.
- Prefer whichever candidate got a given row/column right; you may combine cells from different candidates.
- If both candidates disagree on a cell and you can't tell which is right from the raw text, keep the
  camelot value and add a note in "uncertain_cells".
- Flatten multi-row headers into single column names.
- Preserve a title/description if one is visible in the raw text.

Return ONLY valid JSON (no markdown fences) shaped as:
{{
  "tables": [
    {{
      "title": "...",
      "columns": ["..."],
      "rows": [["...", "..."]],
      "notes": ["..."],
      "uncertain_cells": ["row 3, col 'Total': camelot=120 vs pdfplumber=170, kept camelot"]
    }}
  ]
}}
If there is genuinely no table on this page, return {{"tables": []}}."""


def validate_page(page_num: int, candidates: Dict[str, List[Dict]], text: str) -> Dict[str, Any]:
    prompt = build_validation_prompt(page_num, candidates, text)
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt}],
        max_tokens=4000,
    )
    text_out = resp.choices[0].message.content
    try:
        return json.loads(text_out)
    except json.JSONDecodeError:
        cleaned = re.sub(r"```[a-z]*\n?", "", text_out).strip().rstrip("`")
        return json.loads(cleaned)

## 4. Run the pipeline over the grouped pages

In [11]:
validated_by_page: Dict[int, Dict[str, Any]] = {}

for page_num, candidates in pages_grouped.items():
    try:
        result = validate_page(page_num, candidates, page_text.get(page_num, ""))
    except Exception as e:
        print(f"page {page_num}: validation failed ({e})")
        continue
    validated_by_page[page_num] = result
    n_tables = len(result.get("tables", []))
    print(f"page {page_num}: {n_tables} validated table(s)")

page 1: 1 validated table(s)
page 2: 0 validated table(s)
page 3: 1 validated table(s)
page 5: 0 validated table(s)
page 7: 0 validated table(s)
page 8: 1 validated table(s)
page 9: 0 validated table(s)
page 10: 0 validated table(s)
page 11: 0 validated table(s)
page 12: 1 validated table(s)


## 5. Inspect and save results

In [11]:

# Quick look at one page's validated table(s)
sample_page = next(iter(validated_by_page), None)
if sample_page is not None:
    for tbl in validated_by_page[sample_page].get("tables", []):
        print(tbl.get("title"))
        display(pd.DataFrame(tbl["rows"], columns=tbl["columns"]))
        if tbl.get("uncertain_cells"):
            print("Uncertain:", tbl["uncertain_cells"])

SDA INDIA INDEX 2023-24


,Description
0,INDIA
1,SDG INDIA
2,INDEX 2023-24
3,TOWARDS VIKSIT BHARAT
4,"SUSTAINABLE PROGRESS, INCLUSIVE GROWTH"


In [12]:
page_range_label = f"{min(TARGET_PAGES)}-{max(TARGET_PAGES)}" if TARGET_PAGES else "none"
out_path = OUTPUT_DIR / f"validated_tables_pages_{page_range_label}.json"
with open(out_path, "w") as f:
    json.dump(validated_by_page, f, indent=2, default=str)
print(f"Saved {len(validated_by_page)} page(s) of validated tables to {out_path}")

Saved 10 page(s) of validated tables to data/sda_india_extraction/validated_tables_pages_1-12.json
